# 1.01 — Values, variables and types — solutions

Lesson: [`lessons/1.01_values_and_types.md`](../lessons/1.01_values_and_types.md)

Run the setup cell below first. For every problem: plan in comments (5-step routine), write the code in the stub cell, then run the check cell.

> **Reference solutions.** Open only after you have solved the problem yourself or climbed the hint ladder. Compare, don't copy: then re-solve from blank in a few days.

In [ ]:
# ▶ Run this cell first. It defines check() and expect(), which test your answers,
#   and DATA, the path to the course's shared data folder.
import copy, math
from pathlib import Path

DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "README.md").exists()), Path("data"))

def _same(a, b):
    if hasattr(b, "shape") and hasattr(b, "dtype"):          # numpy arrays
        import numpy as np
        try:
            return np.shape(a) == np.shape(b) and np.allclose(a, b, rtol=1e-5, atol=1e-8)
        except TypeError:
            return False
    if isinstance(b, float) and isinstance(a, (int, float)) and not isinstance(a, bool):
        return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    if isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(b, dict):
        return isinstance(a, dict) and a.keys() == b.keys() and all(_same(a[k], b[k]) for k in b)
    return a == b

def check(fn, cases):
    """Run fn on each (args, expected) pair and report. args is a tuple of arguments."""
    ok = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        if len(shown) > 80:
            shown = shown[:77] + "..."
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ {fn.__name__}({shown}) raised {type(e).__name__}: {e}")
            continue
        if _same(got, expected):
            ok += 1
        else:
            print(f"  ✗ {fn.__name__}({shown}) returned {got!r}, expected {expected!r}")
    status = "✅ all passed" if ok == len(cases) else "keep going"
    print(f"{fn.__name__}: {ok}/{len(cases)} passed — {status}")

def expect(description, condition):
    """For checks that aren't a simple function call: expect('mean is ~0', abs(m) < 1e-6)."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        print(f"  ✗ {description} — raised {type(e).__name__}: {e}")
        return
    print(("  ✅ " if passed else "  ✗ ") + description + ("" if passed else " — keep going"))


## Problem 1 — predict, then check

For each expression, replace `...` with what you *predict* Python will give. Don't run anything first! Then run the check cell and, for each miss, write a one-line comment explaining why.

In [ ]:
my_predictions = {
    "0.1 + 0.2 == 0.3": False,      # binary floats are approximate
    'int("7") + 3': 10,
    "int(3.9)": 3,                  # int() truncates
    "round(3.9)": 4,
    '"5" * 3': "555",               # string repetition
    "True + True": 2,               # bools are ints underneath
    "7 // 2": 3,
    "-7 // 2": -4,                  # floor division rounds DOWN
    "7 % 3": 1,
    "round(2.5)": 2,                # halves round to the even number
    'bool("0")': True,              # any non-empty string is True
    "type(7 / 7).__name__": "float",# / always gives a float
}

In [ ]:
for expr, guess in my_predictions.items():
    actual = eval(expr)
    expect(f"{expr}  →  you said {guess!r}, Python says {actual!r}", guess == actual and type(guess) == type(actual))

## Problem 2 — format a duration

`format_duration(seconds)` returns a string like `"2h 3m 4s"`. Use only `//` and `%`.

In [ ]:
def format_duration(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours}h {minutes}m {secs}s"

In [ ]:
check(format_duration, [
    ((7384,), "2h 3m 4s"),
    ((59,), "0h 0m 59s"),
    ((3600,), "1h 0m 0s"),
    ((86399,), "23h 59m 59s"),
])

## Problem 3 — format money

`format_lkr(amount)` returns the amount with a thousands separator and exactly two decimals, prefixed by `LKR `.

In [ ]:
def format_lkr(amount):
    return f"LKR {amount:,.2f}"

In [ ]:
check(format_lkr, [
    ((1234.5,), "LKR 1,234.50"),
    ((1000000,), "LKR 1,000,000.00"),
    ((0.456,), "LKR 0.46"),
])

## Problem 4 — simple interest

`simple_interest(principal, annual_rate, years)` returns the interest earned (not the final balance), rounded to 2 decimals. `annual_rate` is a fraction, e.g. `0.12` for 12%.

In [ ]:
def simple_interest(principal, annual_rate, years):
    return round(principal * annual_rate * years, 2)

In [ ]:
check(simple_interest, [
    ((100_000, 0.12, 2), 24000.0),
    ((50_000, 0.085, 3), 12750.0),
    ((1234, 0.1, 0.5), 61.7),
])

## Problem 5 — compound interest

`compound_balance(principal, annual_rate, years, periods_per_year)` returns the final balance, rounded to 2 decimals, using

$$ B = P\left(1 + \frac{r}{m}\right)^{m\,t} $$

In [ ]:
def compound_balance(principal, annual_rate, years, periods_per_year):
    rate_per_period = annual_rate / periods_per_year
    n_periods = periods_per_year * years
    return round(principal * (1 + rate_per_period) ** n_periods, 2)

In [ ]:
check(compound_balance, [
    ((100_000, 0.10, 5, 12), 164530.89),
    ((100_000, 0.10, 5, 1), 161051.0),
    ((1000, 0.0, 10, 12), 1000.0),
])

## Problem 6 ★ — split the bill

`split_bill(total, people, tip_pct)` adds a tip (`tip_pct` is e.g. `10` for 10%), splits the result evenly, and **rounds each share up to the next 10 LKR** so nobody under-pays. Return an `int`. Hint: `math.ceil`.

In [ ]:
import math

def split_bill(total, people, tip_pct):
    with_tip = total * (1 + tip_pct / 100)
    share = with_tip / people
    return math.ceil(share / 10) * 10

In [ ]:
check(split_bill, [
    ((10_000, 4, 10), 2750),
    ((9_999, 3, 0), 3340),
    ((1_200, 1, 0), 1200),
    ((5_000, 3, 12.5), 1880),
])

## Problem 7 ★ — digits without loops

`digit_sum_3(n)` returns the sum of the digits of a **3-digit** number, using `//` and `%` only (no strings, no loops).

In [ ]:
def digit_sum_3(n):
    hundreds = n // 100
    tens = (n // 10) % 10
    ones = n % 10
    return hundreds + tens + ones

In [ ]:
check(digit_sum_3, [((482,), 14), ((100,), 1), ((999,), 27)])